In [0]:
bronze_stream_df = (
    spark.readStream
    .table("catalog_smartfactory.bronze.streaming_iot_telemetry")
    )


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

silver_stream_df = (
    bronze_stream_df
    .filter(col("machine_id").isNotNull())
    .filter(col("event_timestamp").isNotNull())
    .filter(col("temperature").between(0,200))
    .filter(col("vibration").between(0,10))
    .filter(col("pressure").between(0,200))
    .withColumn("temperature_alerts",when(col("temperature")>100,1).otherwise(0))
    .withColumn("vibration_alerts",when(col("vibration")>4,1).otherwise(0))
    .withColumn("pressure_alerts",when(col("pressure")>130,1).otherwise(0))
    .withColumn(
        "health_score",
                round(
                    lit(100)
                    - col("temperature_alerts")*20
                    - col("vibration_alerts")*30
                    - col("pressure_alerts")*20
                    - col("failure_risk_score")*30
                    ,2)
                )
    .withColumn(
        "machine_status",
        when(col("health_score") >= 90, "HEALTHY")
        .when(col("health_score") >= 70, "WARNING")
        .otherwise("CRITICAL")
                )
)

In [0]:
silver_query = (
    silver_stream_df.writeStream
    .format("delta")
    .outputMode("Append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/catalog_smartfactory/bronze/checkpoints/silver_iot_stream_v1"
    )
    .toTable("catalog_smartfactory.silver.streaming_iot_telemetry")

)

In [0]:
silver_query.stop()

In [0]:
display(spark.table("catalog_smartfactory.silver.streaming_iot_telemetry"))

timestamp,machine_id,temperature,vibration,pressure,rpm,power_consumption,failure_risk_score,event_timestamp,ingestion_timestamp,temperature_alerts,vibration_alerts,pressure_alerts,health_score,machine_status
2026-08-09T11:48:52.824Z,MCH-1011,103.69561520556262,4.133271723260657,108.521686860259,4276.260803647834,null,0.6570760898626467,2026-08-09T11:48:52.824Z,2026-08-09T11:52:59.014Z,1,1,0,30.29,CRITICAL
2026-08-09T11:48:52.825Z,MCH-1027,72.46197629378001,0.7702474273150259,116.98312837334942,3573.3000505483897,null,0.1700455909231223,2026-08-09T11:48:52.825Z,2026-08-09T11:52:59.014Z,0,0,0,94.9,HEALTHY
2026-08-09T11:48:52.825Z,MCH-1037,79.24678983996337,6.676756803295279,126.52077782705264,3100.5450513099404,null,0.41167312987997734,2026-08-09T11:48:52.825Z,2026-08-09T11:52:59.014Z,0,1,0,57.65,CRITICAL
2026-08-09T11:48:52.826Z,MCH-1049,95.39054727556612,3.903215988420502,139.24841670435924,2476.5082543451776,null,0.4204508818692334,2026-08-09T11:48:52.826Z,2026-08-09T11:52:59.014Z,0,0,1,67.39,CRITICAL
2026-08-09T11:48:52.826Z,MCH-1012,94.3773824987248,4.4190153219506545,131.26518621388277,3567.2911888969093,null,0.6461398958207755,2026-08-09T11:48:52.826Z,2026-08-09T11:52:59.014Z,0,1,1,30.62,CRITICAL
2026-08-09T11:48:52.827Z,MCH-1023,88.31666583959904,5.3106803408022865,119.4297182643611,2523.672553862705,null,0.4493938392835787,2026-08-09T11:48:52.827Z,2026-08-09T11:52:59.014Z,0,1,0,56.52,CRITICAL
2026-08-09T11:48:52.828Z,MCH-1014,74.77522080935347,1.2807610880779008,100.79385551009774,3737.657712451433,null,0.0075340167440672845,2026-08-09T11:48:52.828Z,2026-08-09T11:52:59.014Z,0,0,0,99.77,HEALTHY
2026-08-09T11:48:52.828Z,MCH-1035,65.47687763143935,6.4737839476938515,136.15573491973888,1508.5224487949663,null,0.5553540994221844,2026-08-09T11:48:52.828Z,2026-08-09T11:52:59.014Z,0,1,1,33.34,CRITICAL
2026-08-09T11:48:52.828Z,MCH-1014,90.43027337317329,4.359510405214978,100.51957894394786,2680.211374841081,null,0.5179677688157549,2026-08-09T11:48:52.828Z,2026-08-09T11:52:59.014Z,0,1,0,54.46,CRITICAL
2026-08-09T11:48:52.829Z,MCH-1002,60.07706185078193,1.4493329061709943,85.78068225739199,4024.5310027644755,null,0.13872097211185686,2026-08-09T11:48:52.829Z,2026-08-09T11:52:59.014Z,0,0,0,95.84,HEALTHY
